# Reversion
Se eliminan las tablas de golden, silver y bronze, sus carpetas en el Data Lake (incluido el checkpoint de Auto Loader) y los schemas.
Opcionalmente tambien se elimina el catalogo. Los archivos de la capa raw no se modifican.

In [ ]:
dbutils.widgets.text("catalogo", "catalog_dev")
dbutils.widgets.text("storage_name", "")
dbutils.widgets.dropdown("eliminar_catalogo", "no", ["no", "si"])
dbutils.widgets.dropdown("confirmar", "no", ["no", "si"])

CATALOGO = dbutils.widgets.get("catalogo")
STORAGE = dbutils.widgets.get("storage_name")
ELIMINAR_CATALOGO = dbutils.widgets.get("eliminar_catalogo") == "si"

if dbutils.widgets.get("confirmar") != "si" or not STORAGE:
    raise Exception("Completar storage_name y seleccionar confirmar = 'si'")

In [ ]:
TABLAS = {
    "golden": ["kpi_departamentos", "top_productos", "actividad_dia_hora", "segmentos_clientes"],
    "silver": ["productos", "ordenes", "detalle_ordenes"],
    "bronze": ["orders", "order_products", "products", "aisles", "departments"],
}

def ruta(capa, sub):
    return f"abfss://{capa}@{STORAGE}.dfs.core.windows.net/{CATALOGO}/{sub}"

In [ ]:
for capa, tablas in TABLAS.items():
    for tabla in tablas:
        spark.sql(f"DROP TABLE IF EXISTS {CATALOGO}.{capa}.{tabla}")
        try:
            dbutils.fs.rm(ruta(capa, tabla), True)
        except Exception as e:
            print(f"No se encontro la ruta {ruta(capa, tabla)}: {e}")
        print(f"Eliminada: {CATALOGO}.{capa}.{tabla}")

# Si no se borra el checkpoint, Auto Loader no vuelve a leer los archivos
dbutils.fs.rm(ruta("bronze", "_checkpoints"), True)
print("Checkpoints eliminados")

In [ ]:
for capa in TABLAS:
    spark.sql(f"DROP SCHEMA IF EXISTS {CATALOGO}.{capa} CASCADE")
    print(f"Schema eliminado: {CATALOGO}.{capa}")

if ELIMINAR_CATALOGO:
    spark.sql(f"DROP CATALOG IF EXISTS {CATALOGO} CASCADE")
    print(f"Catalogo eliminado: {CATALOGO}")